# Titanic
## Score: .80382


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd()
DATA = ROOT / "titanic"
RARE = {"Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"}

In [2]:
def build_xy(train_df: pd.DataFrame, test_df: pd.DataFrame):
    y = train_df["Survived"].to_numpy()
    tr = train_df.drop(columns=["Survived"]).copy()
    te = test_df.copy()

    ticket_vc = tr["Ticket"].value_counts()
    med_fare = tr["Fare"].median()
    mode_emb = tr["Embarked"].mode().iloc[0]
    tr["Fare"] = tr["Fare"].fillna(med_fare)
    te["Fare"] = te["Fare"].fillna(med_fare)
    tr["Embarked"] = tr["Embarked"].fillna(mode_emb)
    te["Embarked"] = te["Embarked"].fillna(mode_emb)

    def impute_age(ref: pd.DataFrame, df: pd.DataFrame) -> None:
        gmed = ref["Age"].median()
        for (sex, pcl), m in ref.groupby(["Sex", "Pclass"])["Age"].median().items():
            mm = m if pd.notna(m) else gmed
            msk = (df["Sex"] == sex) & (df["Pclass"] == pcl) & df["Age"].isna()
            df.loc[msk, "Age"] = mm
        df["Age"] = df["Age"].fillna(gmed)

    impute_age(tr, tr)
    impute_age(tr, te)

    def feats(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
        out["Title"] = out["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
        out["Title"] = out["Title"].where(~out["Title"].isin(RARE), "Rare")
        out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
        out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
        out["Deck"] = out["Cabin"].apply(lambda x: str(x)[0] if pd.notna(x) and str(x) and str(x)[0].isalpha() else "U")
        out["TicketGroupSize"] = out["Ticket"].map(ticket_vc).fillna(1).astype(int)
        out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
        out["HasCabin"] = out["Cabin"].notna().astype(int)
        out["AgeBin"] = pd.cut(out["Age"], bins=[0, 12, 18, 35, 60, 200], labels=["c", "t", "y", "a", "s"]).astype(str)
        out["Surname"] = out["Name"].str.split(",").str[0].str.strip()
        out["TicketPrefix"] = out["Ticket"].astype(str).str.replace(r"[0-9./ ]", "", regex=True)
        out["TicketPrefix"] = out["TicketPrefix"].replace("", "NONE")
        out = out.drop(columns=["Name", "Ticket", "Cabin", "PassengerId"], errors="ignore")
        out["Pclass"] = out["Pclass"].astype(str)
        for c in ["Sex", "Embarked", "Title", "Deck", "AgeBin", "Surname", "TicketPrefix"]:
            out[c] = out[c].astype(str)
        return out

    X = feats(tr)
    X_test = feats(te)
    return X, y, X_test


def add_oof_target_encoding(X: pd.DataFrame, y: np.ndarray, X_test: pd.DataFrame, cols, n_splits=5, alpha=20.0):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    y_s = pd.Series(y)
    global_mean = y_s.mean()
    X_new = X.copy()
    X_test_new = X_test.copy()

    for col in cols:
        oof_col = np.zeros(len(X), dtype=float)
        for tr_i, va_i in cv.split(X, y):
            tr_c = X.iloc[tr_i][col]
            y_tr = y_s.iloc[tr_i]
            stat = y_tr.groupby(tr_c).agg(["mean", "count"])
            smooth = (stat["mean"] * stat["count"] + global_mean * alpha) / (stat["count"] + alpha)
            oof_col[va_i] = X.iloc[va_i][col].map(smooth).fillna(global_mean).to_numpy()
        full_stat = y_s.groupby(X[col]).agg(["mean", "count"])
        full_smooth = (full_stat["mean"] * full_stat["count"] + global_mean * alpha) / (full_stat["count"] + alpha)
        X_new[col + "_te"] = oof_col
        X_test_new[col + "_te"] = X_test[col].map(full_smooth).fillna(global_mean).to_numpy()
    return X_new, X_test_new


train_raw = pd.read_csv(DATA / "train.csv")
test_raw = pd.read_csv(DATA / "test.csv")
pid = test_raw["PassengerId"].values
X, y, X_test = build_xy(train_raw, test_raw)
cat_cols = ["Sex", "Embarked", "Title", "Deck", "Pclass", "AgeBin", "Surname", "TicketPrefix"]
te_cols = ["Title", "Deck", "Embarked", "Pclass", "AgeBin", "Surname", "TicketPrefix"]
X_te, X_test_te = add_oof_target_encoding(X, y, X_test, te_cols, n_splits=5, alpha=15.0)

hgb_features = [
    "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "TicketGroupSize", "FarePerPerson", "HasCabin"
] + [c + "_te" for c in te_cols]

imp = SimpleImputer(strategy="median")
X_hgb = imp.fit_transform(X_te[hgb_features])
X_test_hgb = imp.transform(X_test_te[hgb_features])

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_cb1 = np.zeros(len(y), dtype=float)
oof_cb2 = np.zeros(len(y), dtype=float)
oof_hgb = np.zeros(len(y), dtype=float)

p_cb1_t = np.zeros(len(X_test), dtype=float)
p_cb2_t = np.zeros(len(X_test), dtype=float)
p_hgb_t = np.zeros(len(X_test), dtype=float)

fold_acc_cb1 = []
fold_acc_cb2 = []
fold_acc_hgb = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
    Xc_tr, Xc_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    Xh_tr, Xh_va = X_hgb[tr_idx], X_hgb[va_idx]

    cb1 = CatBoostClassifier(
        iterations=20000,
        learning_rate=0.016,
        depth=7,
        l2_leaf_reg=3.0,
        random_seed=SEED + fold,
        thread_count=1,
        verbose=False,
    )
    cb1.fit(
        Xc_tr,
        y_tr,
        cat_features=cat_cols,
        eval_set=(Xc_va, y_va),
        early_stopping_rounds=320,
        verbose=False,
    )

    cb2 = CatBoostClassifier(
        iterations=26000,
        learning_rate=0.012,
        depth=9,
        l2_leaf_reg=4.5,
        random_seed=SEED + 100 + fold,
        thread_count=1,
        verbose=False,
    )
    cb2.fit(
        Xc_tr,
        y_tr,
        cat_features=cat_cols,
        eval_set=(Xc_va, y_va),
        early_stopping_rounds=380,
        verbose=False,
    )

    hgb = HistGradientBoostingClassifier(
        learning_rate=0.02,
        max_iter=1400,
        max_depth=8,
        min_samples_leaf=6,
        l2_regularization=0.35,
        random_state=SEED + fold,
    )
    hgb.fit(Xh_tr, y_tr)

    p1 = cb1.predict_proba(Xc_va)[:, 1]
    p2 = cb2.predict_proba(Xc_va)[:, 1]
    p3 = hgb.predict_proba(Xh_va)[:, 1]

    oof_cb1[va_idx] = p1
    oof_cb2[va_idx] = p2
    oof_hgb[va_idx] = p3

    p_cb1_t += cb1.predict_proba(X_test)[:, 1] / cv.n_splits
    p_cb2_t += cb2.predict_proba(X_test)[:, 1] / cv.n_splits
    p_hgb_t += hgb.predict_proba(X_test_hgb)[:, 1] / cv.n_splits

    fold_acc_cb1.append(accuracy_score(y_va, (p1 >= 0.5).astype(int)))
    fold_acc_cb2.append(accuracy_score(y_va, (p2 >= 0.5).astype(int)))
    fold_acc_hgb.append(accuracy_score(y_va, (p3 >= 0.5).astype(int)))

print("cv_base_acc", {
    "cb1": round(float(np.mean(fold_acc_cb1)), 5),
    "cb2": round(float(np.mean(fold_acc_cb2)), 5),
    "hgb": round(float(np.mean(fold_acc_hgb)), 5),
})

best_acc = -1.0
best_w1, best_w2, best_t = 0.5, 0.2, 0.5
for w1 in np.linspace(0.25, 0.85, 61):
    for w2 in np.linspace(0.05, 0.55, 51):
        if w1 + w2 >= 0.97:
            continue
        w3 = 1.0 - w1 - w2
        p = w1 * oof_cb1 + w2 * oof_cb2 + w3 * oof_hgb
        for t in np.linspace(0.42, 0.62, 101):
            a = accuracy_score(y, (p >= t).astype(int))
            if a > best_acc:
                best_acc = a
                best_w1, best_w2, best_t = float(w1), float(w2), float(t)

best_w3 = 1.0 - best_w1 - best_w2
oof_blend = best_w1 * oof_cb1 + best_w2 * oof_cb2 + best_w3 * oof_hgb
print("cv_blend_acc", round(best_acc, 5), "w", [round(best_w1, 3), round(best_w2, 3), round(best_w3, 3)], "t", round(best_t, 3))
print(confusion_matrix(y, (oof_blend >= best_t).astype(int)))

cv_base_acc {'cb1': 0.84061, 'cb2': 0.84398, 'hgb': 0.82827}
cv_blend_acc 0.85185 w [0.26, 0.54, 0.2] t 0.532
[[508  41]
 [ 91 251]]


In [4]:
blend_t = best_w1 * p_cb1_t + best_w2 * p_cb2_t + best_w3 * p_hgb_t
submission = pd.DataFrame({"PassengerId": pid, "Survived": (blend_t >= best_t).astype(int)})
submission.to_csv(ROOT / "submission.csv", index=False)
print("cv_ensemble", "w", [round(best_w1, 3), round(best_w2, 3), round(best_w3, 3)], "t", round(best_t, 3), ROOT / "submission.csv")
submission.head(10)

cv_ensemble w [0.26, 0.54, 0.2] t 0.532 c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0
